# Code-Übung 3: Finite-Volumen-Methode

Themen dieser Übung sind:
* Diskretisieren der Gleichung für die Finite-Volumen-Methode
    * Unterschied zwischen Finiten Differenzen und Finiten Volumen
    * Approximation des konvektiven Terms mit UDS/CDS
    * Approximation des diffusiven Terms mit CDS
* Verfahrenseigenschaften
    * Konservativität
    * Stabilität

Wir betrachten die stationäre Konvektions-Diffusionsgleichung
$$
\frac{\partial}{\partial x_i}(\rho c_p v_i \phi - \lambda \frac{\partial \phi}{\partial x_i}) = f(x_i)\\
$$
Auf dem rechteckigen 2D-Gebiet $\mathbf{x} = [x,y]$, $x \in [0,L_x]$, $y \in [0,L_y]$.

Dabei beschreibt $\phi$ die Temperatur, $\rho$ die Massendichte, $c_p$ die spezifische Wärmekapazität und $\lambda$ die Wärmeleitfähigkeit. Im gesamten Gebiet sei eine laminare Strömung mit konstantem Vektor der Strömungsgeschwindigkeit $\mathbf{v}$ vorgegeben. Die rechte Seite $f$ beschreibt eine Wärmequelle.
Als Randbedingungen sind auf dem Dirichlet-Rand $\Gamma_D$ die Temperatur $\bar{\phi}_D$ und auf dem Neumann-Rand $\Gamma_N$ der Wärmefluss $\bar{F}_N$ vorgegeben:
$$
\begin{align*}
\phi \vert_{\Gamma_D} &= \bar{\phi}_D\\
-(\rho c_p v_i \phi - \lambda \frac{d \phi}{d x_i})\vert_{\Gamma_N} &= \bar{F}_N
\end{align*}
$$

Das Randwertproblem soll mit der Finite-Volumen-Methode (FVM) numerisch gelöst werden. Hierzu wird die PDGL in die Integralform überführt. Dazu wird das Volumenintegral angewandt, mittels Gauß'schem Integralsatz umgeformt und das Integral auf vier Seiten eines Kontrollvolumens aufgeteilt (siehe Vorlesung):
$$
\sum _c \int _{S_c} (\rho c_p v_i \phi - \lambda \frac{\partial \phi}{\partial x _i}) \boldsymbol{n_{ci}} \, dS_c = \int _V f \, dV
$$
Die entstehende Gleichung wird nun diskretisiert. Dabei werden der konvektive und der diffusive Term separat voneinander betrachtet. Es wird ein **reguläres** Gitter vorausgesetzt. Die Benennung der Seiten erfolgt nach der sogenannten Kompassnotation:

<p align="center"><img src="./graphics/Gitter.png" style="width: 25%;"></p>

Exemplarisch wird der Ostrand betrachtet:
$$
\int _{S_e} (\rho c_p v_1 \phi - \lambda \frac{\partial \phi}{\partial x _1}) \boldsymbol{n_{e1}} \, dS_e = F_e ^C + F_e ^D
$$
Zunächst wird der konvektive Term $F^C$ hergeleitet.  Zum Lösen des Integrals wird die Mittelpunktregel verwendet:
$$
F^C_e = \int _{S_e} (\rho c_p v_i \phi) n_{ei} \, dS_e \approx \rho c_p v_1 \phi _e \cdot \delta S_e
$$
Der Term $\phi _e$ ist der Wert von $\phi$ am rechten Rand des Kontrollvolumens - also zwischen zwei Mittelpunkten benachbarter Kontrollvolumen. Zum bestimmen von $\phi _e$ im konvektiven Term werden zwei mögliche Vorschriften vorgestellt: Das *Central Differencing Scheme* (CDS) und das *Upwind Differencing Scheme* (UDS).

CDS:
$$
\phi _ e = \phi _E \cdot \gamma _e + \phi _P (1- \gamma _e) \\
\gamma _e = \frac{x_e-x_P}{x_E- x_P}
$$

UDS:
$$
\phi_e = \left\{
    \begin{array}{ll}
    \phi_P & \, \textrm{für } \boldsymbol{n_e} \cdot \rho c_p v > 0  \\
    \phi_E & \, \textrm{für } \boldsymbol{n_e} \cdot \rho c_p v < 0 \\
    \end{array}
    \right.
$$
Wird die Variante "UDS" verwendet und angenommen, dass $\rho c_p v_1 > 0$ gilt, so entsteht folgender Term:
$$
F_e ^C \approx \rho c_p v_1 \phi _e \cdot \delta S_e = \rho c_p v_1 \phi _P \cdot \Delta y
$$
Die Variante "CDS" liefert folgendes Ergebnis:
$$
F_e ^C \approx \rho c_p v_1 \phi _e \cdot \delta S_e = \frac{1}{2} (\phi _E + \phi _ P) \rho c_p v_1 \cdot \Delta y
$$

Der diffusive Term wird mit der Mittelspunktregel und der Variante "CDS" betrachtet. "CDS" entspricht hier der Zentraldifferenz.
$$
\int _{S_c} -\lambda \frac{\partial \phi}{\partial x _1} \boldsymbol{n_{e1}} \, dS_c \approx - \lambda \frac{\partial \phi _e}{\partial x_1} \cdot \delta S_e \\[2ex]
\textrm{CDS: } \frac{\partial \phi _e}{\partial x_1} \approx \frac{\phi _E - \phi _P}{x_E - x_P}\\[2ex]
F_e ^D \approx - \lambda \frac{\phi _E - \phi _P}{x_E - x_P} \cdot \Delta y
$$

Das gezeigte Vorgehen am Ostrand wird für alle weiteren Himmelsrichtungen durchgeführt. Der Vektor der rechten Seite wird mit der Mittelpunktregel folgendermaßen approximiert:
$$
\int _V f \, dV \approx \Delta x \Delta y f_P
$$
Die resultierende Gleichung wird umgestellt, sodass der Beitrag aller benachbarten Zellen sichtbar ist:
$$
a _S \phi _S + a_W \phi _W + a_P \phi _P + a_E \phi_E + a_N \phi_N = f_P
$$
Die Koeffizienten $a_c$ sind abhängig von der Entscheidung zwischen UDS und CDS für den konvektiven Term und werden daher für die folgenden Beispiele separat dargestellt.

## Import

In [ ]:
import numpy as np
from helper_functions import plot_heat

## Modellparameter

In [ ]:
# Model parameters
L_x = 1.0                   # x dimension of the domain in m
L_y = 1.0                   # y dimension of the domain in m
rho = 10.0                 # Density of the fluid in kg/m^3
c_p  = 10.0               # heat capacity in J/(kg K)
lambd   = 1.0              # thermal conductivity in W/(m K)
alpha = lambd/(rho*c_p)     # thermal diffusivity in m^2/s
v_1 = 0.1                   # x speed in m/s
v_2 = 0.0                   # y speed in m/s

source_size = 10            # dimension of source in number of cells per side
Q_0 = 1000                  # heat flux in W/m2

scheme_type = 'UDS'         # 'CDS' for central difference scheme, 'UDS' for upwind difference scheme

# Grid parameters
N_x = 50                    # Number of cells in x direction
N_y = 50                    # Number of cells in y direction
delta_x = L_x/N_x           # Step size in x direction in m
delta_y = L_y/N_y           # Step size in y direction in m
N = N_x * N_y               # Total degrees of freedom
K = np.zeros((N, N))        # Stiffness matrix
f = np.zeros(N)             # Load vector
node_transform = lambda i, j: j*N_x + i  # Conversion from grid index (i, j) to degree of freedom index (I) ( = row in load vector)

##  Konvektiver Term mit UDS
### Innere Punkte
Ausgehend von der Gleichung 
$$
a _S \phi _S + a_W \phi _W + a_P \phi _P + a_E \phi_E + a_N \phi_N = f_P
$$
sowie der oben gezeigten Approximation des konvektiven Terms mit UDS, ergeben sich die Koeffizienten $a_c$ als:
$$
\begin{align*}
a_S &= - \frac{\rho c_p v_2}{\Delta y} - \frac{\lambda}{\Delta y ^2}\\[2ex]
a_W &= - \frac{\rho c_p v_1}{\Delta x} - \frac{\lambda}{\Delta x ^2}\\[2ex]
a_P &= \frac{\rho c_p v_1}{\Delta x} + \frac{2 \lambda}{\Delta x ^2} + \frac{\rho c_p v_2}{\Delta y} + \frac{2 \lambda}{\Delta y ^2}\\[2ex]
a_E &= -\frac{\lambda}{\Delta x^2}\\[2ex]
a_N &= -\frac{\lambda}{\Delta y^2}
\end{align*}
$$
Für diese Rechnung wurde jedoch bei der Anwendung des UDS $\rho c_p v_1 > 0$ und $\rho c_p v_2 > 0$ angenommen. Eine geläufige, für beide Strömungsrichtungen funktionierende UDS-Implementierung ist:
$$
\begin{align*}
a_S &= - \textrm{max}(0, \frac{\rho c_p v_2}{\Delta y}) - \frac{\lambda}{\Delta y ^2}\\[2ex]
a_W &= -  \textrm{max}(0, \frac{\rho c_p v_1}{\Delta x}) - \frac{\lambda}{\Delta x ^2}\\[2ex]
a_N &= - \textrm{max}(0, -\frac{\rho c_p v_2}{\Delta y}) - \frac{\lambda}{\Delta y^2}\\[2ex]
a_E &= - \textrm{max}(0, -\frac{\rho c_p v_1}{\Delta x}) - \frac{\lambda}{\Delta x^2}\\[2ex]
a_P &= a_E + a_W + a_N + a_S
\end{align*}
$$

In [ ]:
if scheme_type == 'UDS':
    # Abbreviations to simplify the code for the coefficients:
    dfs_x = lambd / (delta_x**2)
    dfs_y = lambd / (delta_y**2)
    adv_x = rho * c_p * v_1 / delta_x
    adv_y = rho * c_p * v_2 / delta_y

    # Inner nodes
    a_S = dfs_y + max(0, adv_y)
    a_W = dfs_x + max(0, adv_x)
    a_E = dfs_x + max(0, -adv_x)
    a_N = dfs_y + max(0, -adv_y)
    a_P = a_W + a_E + a_S + a_N

### Randbedingungen
Nachdem die inneren Punkte mittels UDS betrachtet wurden, müssen nun die gegebenen Randbedingungen berücksichtigt werden. Hierfür werden die Kontrollvolumina am Rand des Gebiets separat betrachtet.

<p align="center"> <img src="./graphics/Rand.png" style="width: 20%;"> </p>

#### Neumann-Randbedingungen
Die Neumann-Randbedingung hat am westlichen Rand des Gebiets die Form 
$$
\frac{\partial \phi}{\partial x_1} \Big\vert _w = \bar F_w
$$
Ausgehend von der anfänglichen Umformung durch den Satz von Gauß
$$
\sum _c \int _{S_c} (\rho c_p v_i \phi - \lambda \frac{\partial \phi}{\partial x _i}) \boldsymbol{n_{ci}} \, dS_c = \int _V f \, dV
$$
werden nun die zuvor gelösten Integrale erneut betrachtet. Als Beipiel wird diesmal der westliche Rand genommen.

Für den **konvektiven** Term werden nun werden wieder zwei Fälle unterschieden, ausgehend von:
$$
\int _{S_w} (\rho c_p v_i \phi) n_{wi} \, dS_w \approx \rho c_p v_1 \phi _w \cdot \delta S_w
$$
 - Strömung von Westen nach Osten: Unter Benutzung des gegebenen Stroms am Rand, wird der Wert für $\phi_w$ durch eine lineare Interpolation genähert. Dabei ist insbesondere zu beachten, dass der Abstand zwischen dem Mittelpunkt eines Kontrollvolumens und dem Rand des Gebiets nur halb so groß ist, wie der Abstand zweier Mittelpunkte (vgl. Abbildung).
    $$
        F_w ^C \approx \rho c_p v_1 \phi _w \cdot \delta S_w\\[2ex]
        = \rho c_p v_1 \Big(\phi_P - \frac{\partial \phi}{\partial x_1} \vert_w \cdot (x_P-x_w)\Big) \cdot (y _{nw} -y _{sw})\\[2ex]
        = \rho c_p v_1 \Big(\phi_P - \bar F_w \cdot (x_P-x_w)\Big) \cdot (y _{nw} -y _{sw})
    $$
 - Strömung von Osten nach Westen: Es wird $\phi _w = \phi _P$ gesetzt.
        $$
        F_w ^C \approx \rho c_p v_1 \phi _w \cdot \delta S_w\\[2ex]
        = \rho c_p v_1 \phi_P \cdot (y _{nw} -y _{sw})
        $$

Für die Herleitung des **diffusiven** Terms kann die gegebene Randbedingung direkt eingesetzt werden.  

$$
\int _{S_c} -\lambda \frac{\partial \phi}{\partial x _1} \boldsymbol{n_{w1}} \, dS_c = - \lambda \frac{\partial \phi}{\partial x_1}\vert _w \cdot \delta S_w = - \lambda \bar F_w \cdot \delta S_e
$$

Für den Fall einer Strömung von **Westen nach Osten** erhält man durch Umstellen der Gleichung in die Form
$$
a _S \phi _S + a_W \phi _W + a_P \phi _P + a_E \phi_E + a_N \phi_N = f_P
$$
für $a_N$, $a_E$, $a_S$, $a_P$ die selben Ergebnisse wie bereits für innere Punkte. $a_W$ und $f_P$ werden neu bestimmt: 
    $$
    f_P = f\vert_P + \frac{\bar F_w}{x_e-x_w}\\[2ex]
    a_W = 0
    $$

#### Dirichlet-Randbedingungen
Um Dirichlet-Randbedingungen der Form
$$
\phi \vert _w = \bar\phi_w
$$
zu verwenden, ist die Herleitung sehr ähnlich zu den Neumann-Randbedingungen und wird daher nicht im Detail vorgestellt. Auch hier wird der Einfachheit nur eine Strömung von **Westen nach Osten** betrachtet. Die Koeffizienten $a_N$, $a_E$, $a_S$ sind identisch zu den zuvor für innere Knoten berechneten Werten, da auch hier $v_1 > 0$ angenommen wurde. $a_W$ und $a_P$, sowie $f_P$ werden neu bestimmt: 
    $$
    a_W = 0\\[2ex]
    a_P = \frac{\rho c_p v_1}{\Delta x} + \frac{\lambda}{\Delta x^2} + \frac{\rho c_p v_2}{\Delta y} + \frac{\lambda}{\Delta y^2}\\[2ex]
    f_P = f\vert_P + [\frac{\rho c_p v_1}{\Delta x}+\frac{\lambda}{2 \Delta x^2}]\bar\phi_w
    $$

Für die Randbedingungen kann wieder analog zu den inneren Punkten eine Schreibweise mit dem $\max()$ Operator hergeleitet werden. Diese wird in der Implementierung verwendet.

In [ ]:
if scheme_type == 'UDS':
    # Dirichlet boundary condition coefficients
    a_P_dirichlet_W = 2*dfs_x + max(0, adv_x)
    a_P_dirichlet_E = 2*dfs_x + max(0, -adv_x)
    a_P_dirichlet_S = 2*dfs_y + max(0, adv_y)
    a_P_dirichlet_N = 2*dfs_y + max(0, -adv_y)

## Konvektiver Term mit CDS
Einsetzen in die Formel vom Anfang führt auf:
$$
\begin{align*}
a_S &= -\frac{\rho c_p v_2}{2 \Delta y}-\frac{\lambda}{\Delta y ^2}\\[2ex]
a_W &= -\frac{\rho c_p v_1}{2 \Delta x}-\frac{\lambda}{\Delta x ^2}\\[2ex]
a_P &= \frac{\rho c_p v_1}{\Delta x}+\frac{2 \lambda}{\Delta x ^2}+\frac{\rho c_p v_2}{\Delta y}+\frac{2 \lambda}{\Delta y ^2}\\[2ex]
a_E &= -\frac{\rho c_p v_1}{2 \Delta x}-\frac{\lambda}{\Delta x ^2}\\[2ex]
a_N &= -\frac{\rho c_p v_2}{2 \Delta y}-\frac{\lambda}{\Delta y ^2}\\
\end{align*}
$$

In [ ]:
if scheme_type == 'CDS':
    # Abbreviations to simplify the code for the coefficients of the stiffness matrix
    dfs_x = lambd / (delta_x**2)
    dfs_y = lambd / (delta_y**2)
    adv_x = rho * c_p * v_1 / (2 * delta_x)
    adv_y = rho * c_p * v_2 / (2 * delta_y)

    # Inner nodes
    a_S = dfs_y + adv_y
    a_W = dfs_x + adv_x
    a_E = dfs_x - adv_x
    a_N = dfs_y - adv_y
    a_P = a_W + a_E + a_S + a_N

### Randbedingungen
Die Herleitung der Randbedingungen erfolgt, abgesehen von der Fallentscheidung, analog zum UDS und wird daher nicht erneut gezeigt.

In [ ]:
if scheme_type == 'CDS':
    # Dirichlet boundary condition coefficients
    a_P_dirichlet_W = 2*dfs_x + adv_x
    a_P_dirichlet_E = 2*dfs_x - adv_x
    a_P_dirichlet_S = 2*dfs_y + adv_y
    a_P_dirichlet_N = 2*dfs_y - adv_y

## Aufstellen des Gleichungssystems für innere Punkte
Anmerkungen zum gezeigten Vorgehen:
 - Da die Randbedingungen am Rand des betrachteten Gebiets lediglich die Koeffizienten $a_P$ und den seitenspezifischen Koeffizienten (z.B. $a_W$) betreffen, werden die Kontrollvolumina am Rand hier zunächst wie innere Punkte behandelt, wobei die seitenspezifischen Koeffizienten, die $a=0$ sind, ausgelassen werden.
 - Im zweiten Schritt wird die Koeffizientenmatrix ergänzt, wobei die durch die Randbedingungen geänderten Koeffizienten $a_P$ und $f_P$ ergänzt werden.

In [ ]:
# Check if the scheme type is valid and coefficients have been defined
if scheme_type != 'CDS' and scheme_type != 'UDS':
    raise ValueError('scheme_type must be either "CDS" or "UDS".')

# loop over cells
for j in range(N_y):
    for i in range(N_x):

        p = node_transform(i, j)

        # Disregard a_W on western boundary
        if i > 0:
            W = node_transform(i-1, j)
            K[p, W] = -a_W
            K[p, p] += a_W

        # east boundary
        if i < N_x-1:
            E = node_transform(i+1, j)
            K[p, E] = -a_E
            K[p, p] += a_E

        # south boundary
        if j > 0:
            S = node_transform(i, j-1)
            K[p, S] = -a_S
            K[p, p] += a_S

        # north boundary
        if j < N_y-1:
            Nn = node_transform(i, j+1)
            K[p, Nn] = -a_N
            K[p, p] += a_N


# Assemble the RHS vector (here: Heat source)
# Define heat source in spatial coordinates first (dimension N_x x N_y)
heat_source = np.zeros((N_x, N_y))
heat_source[round(N_x/2-source_size/2):round(N_x/2+source_size/2), round(N_y/2-source_size/2):round(N_y/2+source_size/2)] = Q_0

# Flattening the heat source to be a vector (each row now corresponds to one element)
f = heat_source.flatten(order = 'F')

## Einbinden der Randbedingungen

In [ ]:
# bc-order: west, east, south, north
bc_types = ["dirichlet", "dirichlet", "dirichlet", "dirichlet"]
T = [300, 300, 300, 300]
q = [0, 0, 0, 0]

# West Boundary
match bc_types[0]:
    case "dirichlet":
        for j in range(N_y):
            p = node_transform(0, j)
            K[p, p] += a_P_dirichlet_W       # Update a_P for Dirichlet BC
            f[p] += a_P_dirichlet_W * T[0]   # Update load vector for Dirichlet BC
    case "neumann":
        for j in range(N_y):
            p = node_transform(0, j)
            f[p] += q[0] / delta_x           # Update load vector for Neumann BC


# East Boundary
match bc_types[1]:
    case "dirichlet":
        for j in range(N_y):
            p = node_transform(N_x-1, j)
            K[p, p] += a_P_dirichlet_E       
            f[p] += a_P_dirichlet_E * T[1]   
    case "neumann":
        for j in range(N_y):
            p = node_transform(N_x-1, j)
            f[p] += q[1] / delta_x


# South Boundary
match bc_types[2]:
    case "dirichlet":
        for i in range(N_x):
            p = node_transform(i, 0)
            K[p, p] += a_P_dirichlet_S
            f[p] += a_P_dirichlet_S * T[2]
    case "neumann":
        for i in range(N_x):
            p = node_transform(i, 0)
            f[p] += q[2] / delta_y


# North Boundary
match bc_types[3]:
    case "dirichlet":
        for i in range(N_x):
            p = node_transform(i, N_y-1)
            K[p, p] += a_P_dirichlet_N
            f[p] += a_P_dirichlet_N * T[3]
    case "neumann":
        for i in range(N_x):
            p = node_transform(i, N_y-1)
            f[p] += q[3] / delta_y

## Lösen des Gleichungssystems

In [ ]:
Phi = np.linalg.solve(K, f)

## Plotten der Ergebnisse

In [ ]:
# Reshape to grid
Phi = Phi.reshape((N_y, N_x))
heat_source = heat_source.T


# define cell centers
x = np.linspace(delta_x, L_x - delta_x, N_x)
y = np.linspace(delta_y, L_y - delta_y, N_y)
meshgrid_x, meshgrid_y = np.meshgrid(x, y)

# Plot the temperature field and the heat source for reference
plot_heat(meshgrid_x, meshgrid_y, heat_source, Phi)

### Interpretation der Ergebnisse

Stabilität des Verfahrens:

* Für reine Diffusion, also $v_x = v_y = 0$ ist das Verfahren immer stabil.
* Das Verfahren ist bei Anwendung des UDS-Schemas für den konvektiven Term immer stabil.
* Das Verfahren ist bei Anwendung des CDS-Schemas für den konvektiven Term nur bereichsweise stabil.

Im letzten Fall wird die Grenze ab der Instabilitäten auftreten durch eine Bedingung für die Peclet-Zahl $\mathrm{Pe}_h$ mit allgemeiner Gitterweite $h$ beschrieben.
In diesem Falle muss gelten $\mathrm{Pe}_h~=~\dfrac{c_p\rho v h}{\lambda} = \dfrac{v h}{\alpha} \leq 2 $. 
Um Stabilität sicher zu stellen, folgt für die Wahl der Gitterweiten:
$$
\Delta x \leq \frac{2\alpha}{|v_x|} \;\;\; \mathrm{und} \;\;\; \Delta y \leq \frac{2\alpha}{|v_y|}
$$